# 05 — CTI Entity Extraction on CoDA

Classification tells you *what a page is about*. Extraction tells you *what specific things it names* — the wallet addresses, CVEs, emails, threat-actor aliases, malware families. That's the form of output a SOC pipeline can actually act on.

This notebook does three things:

1. **Compares three labeling strategies** on CoDA — regex, GLiNER zero-shot, and a regex ∪ GLiNER hybrid. Qualitative comparison only: span counts, pairwise agreement, side-by-side examples. (We don't have hand-labeled gold, so we don't report F1 vs. ground truth.)
2. **Builds silver labels on all English CoDA pages** using the hybrid strategy.
3. **Fine-tunes DarkBERT as a token classifier** on those silver labels, with overlapping 512-token chunks so long pages aren't truncated.

## Entity schema

| Entity | Labeler | Notes |
|---|---|---|
| `CRYPTO_WALLET` | regex | BTC / ETH / XMR addresses |
| `EMAIL` | regex | full RFC-ish pattern |
| `ONION_URL` | regex | `.onion` v2/v3 |
| `CVE` | regex | `CVE-YYYY-NNNN` |
| `IP_ADDRESS` | regex | IPv4 only |
| `CREDENTIAL_TYPE` | regex | fullz, CVV, dumps, RDP, SSH, logs, bins |
| `THREAT_ACTOR` | GLiNER | fuzzy — APT28, FIN7, Lazarus |
| `MALWARE_FAMILY` | GLiNER | Cobalt Strike, Emotet, RedLine |
| `ORGANIZATION` | GLiNER | victim orgs, companies mentioned |

## Prerequisites

```bash
pip install -U transformers datasets scikit-learn torch gliner seqeval
```

Requires the CoDA data from nb 02a and Hugging Face access to `s2w-ai/DarkBERT`.

## 1 — Load CoDA English corpus

In [2]:
import re, json, random
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np, pandas as pd
import torch

CODA_DIR = Path('data/coda/coda_dataset')
assert CODA_DIR.exists(), 'Run nb 02a or nb 03 first to fetch CoDA.'

rows = []
for f in CODA_DIR.glob('*.txt'):
    parts = f.stem.split('-')
    if len(parts) < 4 or parts[2] != 'en' or parts[1] == 'Others':
        continue
    text = f.read_text(errors='ignore').strip()
    if len(text.split()) < 20:
        continue
    rows.append({'id': parts[0], 'category': parts[1], 'text': text})

coda = pd.DataFrame(rows)
print(f'Loaded {len(coda)} English CoDA pages ({coda["category"].nunique()} categories)')
print(f'Median length: {int(coda["text"].str.split().str.len().median())} words')

Loaded 6582 English CoDA pages (9 categories)
Median length: 578 words


## 2 — Regex labeler

Deterministic patterns for the entity types that don't need a model. A span is a `(start_char, end_char, label)` tuple — the universal format we'll use across all three labelers.

In [3]:
REGEX_RULES = [
    # order matters — more specific first, so greedier patterns don't swallow specific ones
    ('ONION_URL',       re.compile(r'\b[a-z2-7]{16,56}\.onion\b')),
    ('CVE',             re.compile(r'\bCVE-\d{4}-\d{4,7}\b', re.IGNORECASE)),
    ('CRYPTO_WALLET',   re.compile(r'\b(?:bc1[a-z0-9]{25,60}|[13][a-km-zA-HJ-NP-Z1-9]{25,34}|0x[a-fA-F0-9]{40}|4[0-9AB][1-9A-HJ-NP-Za-km-z]{93})\b')),
    ('EMAIL',           re.compile(r'\b[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}\b')),
    ('IP_ADDRESS',      re.compile(r'\b(?:(?:25[0-5]|2[0-4]\d|[01]?\d?\d)\.){3}(?:25[0-5]|2[0-4]\d|[01]?\d?\d)\b')),
    ('CREDENTIAL_TYPE', re.compile(r'\b(?:fullz|dumps?|cvv2?|rdp|ssh|vpn\s+accounts?|logs?|bins?|track2)\b', re.IGNORECASE)),
]

def regex_spans(text):
    spans = []
    for label, pat in REGEX_RULES:
        for m in pat.finditer(text):
            spans.append((m.start(), m.end(), label))
    return dedupe_overlap(spans)

def dedupe_overlap(spans):
    """Keep longer/earlier spans when two overlap — prevents a CVE matching inside a DOMAIN, etc."""
    spans = sorted(spans, key=lambda s: (s[0], -(s[1] - s[0])))
    kept = []
    last_end = -1
    for s, e, l in spans:
        if s >= last_end:
            kept.append((s, e, l)); last_end = e
    return kept

# Sanity-check on a realistic snippet
demo = ("Fullz for sale: US CC with CVV2, DOB, SSN. Contact tox: 1A1zP1eP5QGefi2DMPTfTL5SLmv7DivfNa or email me@protonmail.com. "
        "Exploit info on CVE-2024-1234 posted at deadc0de.onion, mirror 192.168.1.1.")
for s, e, l in regex_spans(demo):
    print(f'  {l:18s} {demo[s:e]!r}')

  CREDENTIAL_TYPE    'Fullz'
  CREDENTIAL_TYPE    'CVV2'
  CRYPTO_WALLET      '1A1zP1eP5QGefi2DMPTfTL5SLmv7DivfNa'
  EMAIL              'me@protonmail.com'
  CVE                'CVE-2024-1234'
  IP_ADDRESS         '192.168.1.1'


## 3 — GLiNER zero-shot labeler

GLiNER takes arbitrary entity-type *descriptions* as prompts. We give it three fuzzy types that regex can't handle: threat actors, malware families, organizations.

In [4]:
from gliner import GLiNER

gliner = GLiNER.from_pretrained("gliner-community/gliner_large-v2.5")
if torch.cuda.is_available():
    gliner = gliner.to('cuda')

GLINER_LABELS = ['threat_actor', 'malware_family', 'organization']
GLINER_LABEL_MAP = {'threat_actor': 'THREAT_ACTOR',
                    'malware_family': 'MALWARE_FAMILY',
                    'organization': 'ORGANIZATION'}

def gliner_spans(text, threshold=0.5, max_chars=4000):
    # GLiNER has its own internal length cap; chunk long text at char boundaries
    all_spans = []
    for offset in range(0, len(text), max_chars):
        chunk = text[offset:offset + max_chars]
        try:
            ents = gliner.predict_entities(chunk, GLINER_LABELS, threshold=threshold)
        except Exception:
            continue
        for e in ents:
            all_spans.append((offset + e['start'], offset + e['end'], GLINER_LABEL_MAP[e['label']]))
    return dedupe_overlap(all_spans)

# Sanity check
demo2 = ("APT28 and FIN7 continue to deploy Cobalt Strike against energy companies. "
         "Microsoft reported that RedLine stealer targets Fortune 500 firms including Pfizer.")
for s, e, l in gliner_spans(demo2):
    print(f'  {l:16s} {demo2[s:e]!r}')

/home/saqib/venv/lib/python3.12/site-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

  THREAT_ACTOR     'APT28'
  THREAT_ACTOR     'FIN7'
  MALWARE_FAMILY   'Cobalt Strike'
  ORGANIZATION     'energy companies'
  ORGANIZATION     'Microsoft'
  MALWARE_FAMILY   'RedLine'
  ORGANIZATION     'Fortune 500'
  ORGANIZATION     'Pfizer'


## 4 — Hybrid labeler

Union of regex and GLiNER spans. When they overlap, regex wins (it's deterministic and high-precision on the entity types it covers). This is what we'll use to produce the silver training labels.

In [5]:
def hybrid_spans(text):
    r = regex_spans(text)
    g = gliner_spans(text)
    # Regex wins conflicts — drop any GLiNER span overlapping a regex span
    r_ranges = [(s, e) for s, e, _ in r]
    def overlaps_regex(s, e):
        return any(not (e <= rs or s >= re_) for rs, re_ in r_ranges)
    g_clean = [(s, e, l) for s, e, l in g if not overlaps_regex(s, e)]
    return dedupe_overlap(r + g_clean)

## 5 — Qualitative three-way comparison (500 sampled pages)

Span counts per method per entity, pairwise Jaccard overlap, and hand-inspected examples. The point isn't "which is best" — it's "how do they differ?"

In [6]:
SAMPLE_N = 500
sample = coda.sample(SAMPLE_N, random_state=42).reset_index(drop=True)

print(f'Running three labelers on {SAMPLE_N} pages (GLiNER is the slow one — a few minutes on GPU)...')
sample['regex'] = sample['text'].apply(regex_spans)
sample['gliner'] = sample['text'].apply(gliner_spans)
sample['hybrid'] = sample['text'].apply(hybrid_spans)

def counts_by_label(spans_list):
    c = Counter()
    for spans in spans_list:
        for _, _, l in spans:
            c[l] += 1
    return c

cnt_r = counts_by_label(sample['regex'])
cnt_g = counts_by_label(sample['gliner'])
cnt_h = counts_by_label(sample['hybrid'])

all_labels = sorted(set(cnt_r) | set(cnt_g) | set(cnt_h))
print(f'\nSpan counts across {SAMPLE_N} pages:\n')
print(f'  {"Entity":<18s} {"regex":>8s} {"gliner":>8s} {"hybrid":>8s}')
for l in all_labels:
    print(f'  {l:<18s} {cnt_r[l]:>8d} {cnt_g[l]:>8d} {cnt_h[l]:>8d}')

Running three labelers on 500 pages (GLiNER is the slow one — a few minutes on GPU)...


/home/saqib/venv/lib/python3.12/site-packages/gliner/data_processing/processor.py:395: UserWarning: Sentence of length 850 has been truncated to 768
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
/home/saqib/venv/lib/python3.12/site-packages/gliner/data_processing/processor.py:395: UserWarning: Sentence of length 869 has been truncated to 768
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
/home/saqib/venv/lib/python3.12/site-packages/gliner/data_processing/processor.py:395: UserWarning: Sentence of length 872 has been truncated to 768
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
/home/saqib/venv/lib/python3.12/site-packages/gliner/data_processing/processor.py:395: UserWarning: Sentence of length 906 has been truncated to 768
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
/home/sa


Span counts across 500 pages:

  Entity                regex   gliner   hybrid
  CREDENTIAL_TYPE         404        0      404
  CRYPTO_WALLET             5        0        5
  ONION_URL                26        0       26


In [7]:
# --- Pairwise agreement (Jaccard over spans, ignoring label) ---------
def span_set(spans):
    return {(s, e) for s, e, _ in spans}

def jaccard(a, b):
    if not a and not b: return 1.0
    return len(a & b) / max(len(a | b), 1)

jac_rg = np.mean([jaccard(span_set(r), span_set(g))
                  for r, g in zip(sample['regex'], sample['gliner'])])
jac_rh = np.mean([jaccard(span_set(r), span_set(h))
                  for r, h in zip(sample['regex'], sample['hybrid'])])
jac_gh = np.mean([jaccard(span_set(g), span_set(h))
                  for g, h in zip(sample['gliner'], sample['hybrid'])])

print('Mean Jaccard overlap between methods (span position, ignoring label):')
print(f'  regex  ∩ gliner = {jac_rg:.3f}   (expected low — different entity types)')
print(f'  regex  ∩ hybrid = {jac_rh:.3f}   (expected high — hybrid contains regex)')
print(f'  gliner ∩ hybrid = {jac_gh:.3f}   (expected partial — hybrid drops overlapping gliner spans)')

Mean Jaccard overlap between methods (span position, ignoring label):
  regex  ∩ gliner = 0.700   (expected low — different entity types)
  regex  ∩ hybrid = 1.000   (expected high — hybrid contains regex)
  gliner ∩ hybrid = 0.700   (expected partial — hybrid drops overlapping gliner spans)


In [8]:
# --- Side-by-side examples -------------------------------------------
def annotate(text, spans, snippet_len=400):
    """Render `text[:snippet_len]` with spans marked inline as [TEXT](LABEL)."""
    spans = sorted(spans)
    out, cursor = [], 0
    for s, e, l in spans:
        if s >= snippet_len: break
        out.append(text[cursor:s])
        out.append(f'[{text[s:e]}]({l})')
        cursor = e
    out.append(text[cursor:snippet_len])
    return ''.join(out)

# Pick a few pages where all three methods found something
interesting = sample[(sample['regex'].apply(len) > 0) &
                  (sample['gliner'].apply(len) > 0)]




for _, row in interesting.iterrows():
    print(f'===== page id={row["id"]} ({row["category"]}) =====')
    print(f'\n--- REGEX ---\n{annotate(row["text"], row["regex"])}')
    print(f'\n--- GLINER ---\n{annotate(row["text"], row["gliner"])}')
    print(f'\n--- HYBRID ---\n{annotate(row["text"], row["hybrid"])}')
    print()

**What to notice.** Regex is high-precision on its entity types but blind to threat actors / malware / orgs. GLiNER catches the fuzzy entities but sometimes truncates formatted strings (e.g., `noreply@evil.ru` → just `noreply`). The hybrid gets the best of both. The Jaccard overlap numbers make this concrete: regex and GLiNER disagree on most spans because they cover disjoint entity types, not because one is wrong.

## 6 — Build silver labels on the full corpus

Run the hybrid labeler on all CoDA English pages. This is what we'll train the DarkBERT extractor on.

In [9]:
PROCESSED = Path('processed'); PROCESSED.mkdir(exist_ok=True)
SILVER = PROCESSED / 'coda_silver_spans.json'

if SILVER.exists():
    print('Silver labels already built, loading from disk.')
    with open(SILVER) as f:
        silver = json.load(f)
    docs = [(d['id'], d['text'], [tuple(s) for s in d['spans']]) for d in silver]
else:
    print(f'Labeling {len(coda)} pages — takes ~10-20 min on GPU (GLiNER is the bottleneck)...')
    docs = []
    for i, row in coda.iterrows():
        spans = hybrid_spans(row['text'])
        docs.append((row['id'], row['text'], spans))
        if (i + 1) % 500 == 0:
            print(f'  {i+1}/{len(coda)}')
    with open(SILVER, 'w') as f:
        json.dump([{'id': i, 'text': t, 'spans': s} for i, t, s in docs], f)
    print(f'Saved silver labels to {SILVER}')

total_spans = sum(len(s) for _, _, s in docs)
spans_by_label = Counter(l for _, _, spans in docs for _, _, l in spans)
print(f'\nTotal spans: {total_spans}  (avg {total_spans/len(docs):.1f} per page)')
print('Spans by label:')
for l, c in spans_by_label.most_common():
    print(f'  {l:<18s} {c}')

Silver labels already built, loading from disk.

Total spans: 10280  (avg 1.6 per page)
Spans by label:
  CREDENTIAL_TYPE    9825
  ONION_URL          369
  CRYPTO_WALLET      84
  EMAIL              2


## 7 — Convert to BIO-tagged training data with chunking

Standard HF token-classification format: each token gets a BIO tag (`B-CVE`, `I-CVE`, `O`). Long pages (most of CoDA) get split into **overlapping 512-token chunks** via `return_overflowing_tokens=True, stride=64` — this way the extractor sees entities past position 350, which a straight truncation would miss.

In [10]:
from transformers import AutoTokenizer

MODEL_CKPT = 's2w-ai/DarkBERT'
MAX_LEN = 512
STRIDE = 64

tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT, use_fast=True)

# --- Build label vocabulary ------------------------------------------
entity_types = sorted(spans_by_label.keys())
tag_names = ['O'] + [f'{p}-{e}' for e in entity_types for p in ('B', 'I')]
tag2id = {t: i for i, t in enumerate(tag_names)}
id2tag = {i: t for t, i in tag2id.items()}
print(f'{len(tag_names)} BIO tags')

# --- Tokenize + align labels via offset mapping -----------------------
def encode_with_labels(text, spans):
    enc = tokenizer(text,
                    truncation=True, max_length=MAX_LEN,
                    return_offsets_mapping=True,
                    return_overflowing_tokens=True,
                    stride=STRIDE)
    chunks = []
    for ids, offsets in zip(enc['input_ids'], enc['offset_mapping']):
        labels = [-100] * len(ids)  # -100 = ignore in loss (for special tokens)
        for ti, (s, e) in enumerate(offsets):
            if s == e == 0:
                continue  # special token
            labels[ti] = tag2id['O']
            for sp_s, sp_e, sp_l in spans:
                if s >= sp_s and e <= sp_e:
                    prefix = 'B' if s == sp_s else 'I'
                    labels[ti] = tag2id[f'{prefix}-{sp_l}']
                    break
        chunks.append({'input_ids': ids, 'attention_mask': [1]*len(ids), 'labels': labels})
    return chunks

print('Encoding all docs into chunks...')
all_chunks = []
for doc_id, text, spans in docs:
    all_chunks.extend(encode_with_labels(text, spans))
print(f'{len(all_chunks)} training chunks from {len(docs)} pages '
      f'({len(all_chunks)/len(docs):.1f} chunks per page)')

# --- 90/10 train/eval split on chunk level ---------------------------
random.seed(42); random.shuffle(all_chunks)
split = int(0.9 * len(all_chunks))
train_chunks, eval_chunks = all_chunks[:split], all_chunks[split:]

from datasets import Dataset
train_ds = Dataset.from_list(train_chunks)
eval_ds = Dataset.from_list(eval_chunks)
print(f'train: {len(train_ds)}  eval: {len(eval_ds)}')

9 BIO tags
Encoding all docs into chunks...
41226 training chunks from 6582 pages (6.3 chunks per page)
train: 37103  eval: 4123


## 8 — Fine-tune DarkBERT as a token classifier

Standard HF recipe: `AutoModelForTokenClassification` + `DataCollatorForTokenClassification` + `seqeval` metrics. 3 epochs, fp16 for the 8 GB GPU.

In [11]:
from transformers import (AutoModelForTokenClassification, TrainingArguments,
                          Trainer, DataCollatorForTokenClassification)
from seqeval.metrics import precision_score, recall_score, f1_score, classification_report

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CKPT,
    num_labels=len(tag_names),
    id2label=id2tag,
    label2id=tag2id,
)

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = np.argmax(preds, axis=-1)
    true_tags, pred_tags = [], []
    for p_seq, l_seq in zip(preds, labels):
        t, p = [], []
        for pi, li in zip(p_seq, l_seq):
            if li == -100:
                continue
            t.append(id2tag[int(li)])
            p.append(id2tag[int(pi)])
        true_tags.append(t); pred_tags.append(p)
    return {
        'precision': precision_score(true_tags, pred_tags),
        'recall':    recall_score(true_tags, pred_tags),
        'f1':        f1_score(true_tags, pred_tags),
    }

args = TrainingArguments(
    output_dir='models/darkbert_entity',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    fp16=True,
    logging_steps=50,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForTokenClassification LOAD REPORT from: s2w-ai/DarkBERT
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.weight               | MISSING    | 
classifier.bias                 | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/home/saqib/venv/lib/python3.12/site-

Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.004180,0.000344,0.929048,0.957831,0.943220
2,0.000090,0.000291,0.937186,0.962995,0.949915
3,0.000077,0.000199,0.952981,0.976764,0.964726


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=13914, training_loss=0.016355482667821736, metrics={'train_runtime': 2766.8204, 'train_samples_per_second': 40.23, 'train_steps_per_second': 5.029, 'total_flos': 2.908652136526541e+16, 'train_loss': 0.016355482667821736, 'epoch': 3.0})

## 9 — Per-entity evaluation

Eval is against silver labels, not gold — so these numbers tell us *how well the model recovered the weak-supervision signal*, not how well it finds real entities in the absolute sense. Still useful: if the model can't even recover the regex signal, something is broken.

In [12]:
out = trainer.predict(eval_ds)
preds = np.argmax(out.predictions, axis=-1)
true_tags, pred_tags = [], []
for p_seq, l_seq in zip(preds, out.label_ids):
    t, p = [], []
    for pi, li in zip(p_seq, l_seq):
        if li == -100: continue
        t.append(id2tag[int(li)])
        p.append(id2tag[int(pi)])
    true_tags.append(t); pred_tags.append(p)

print('Overall:')
print(f'  precision: {precision_score(true_tags, pred_tags):.4f}')
print(f'  recall:    {recall_score(true_tags, pred_tags):.4f}')
print(f'  f1:        {f1_score(true_tags, pred_tags):.4f}')

print('\nPer-entity (vs. silver labels):')
print(classification_report(true_tags, pred_tags, digits=4))

Overall:
  precision: 0.9530
  recall:    0.9768
  f1:        0.9647

Per-entity (vs. silver labels):
                 precision    recall  f1-score   support

CREDENTIAL_TYPE     0.9955    0.9991    0.9973      1118
  CRYPTO_WALLET     0.6250    0.6250    0.6250         8
      ONION_URL     0.2131    0.3611    0.2680        36

      micro avg     0.9530    0.9768    0.9647      1162
      macro avg     0.6112    0.6617    0.6301      1162
   weighted avg     0.9688    0.9768    0.9722      1162



## 10 — Manual QA + save

Run the extractor on a few pages and read the output. Save the model for nb 06.

In [13]:
FINAL = Path('models/darkbert_entity_final')
FINAL.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(FINAL))
tokenizer.save_pretrained(str(FINAL))
with open(FINAL / 'tag_vocab.json', 'w') as f:
    json.dump({'tags': tag_names, 'tag2id': tag2id}, f, indent=2)
print(f'Saved to {FINAL}')

# --- Inference helper that reassembles BIO tags into spans -----------
def extract_entities(text):
    enc = tokenizer(text, truncation=True, max_length=MAX_LEN,
                    return_offsets_mapping=True, return_tensors='pt').to(model.device)
    offsets = enc.pop('offset_mapping')[0].cpu().numpy()
    model.eval()
    with torch.no_grad():
        logits = model(**enc).logits[0].cpu().numpy()
    tag_ids = logits.argmax(-1)
    spans, cur = [], None
    for (s, e), tid in zip(offsets, tag_ids):
        if s == e == 0: continue
        tag = id2tag[int(tid)]
        if tag == 'O':
            if cur: spans.append(cur); cur = None
        elif tag.startswith('B-'):
            if cur: spans.append(cur)
            cur = [int(s), int(e), tag[2:]]
        elif tag.startswith('I-') and cur and cur[2] == tag[2:]:
            cur[1] = int(e)
        else:  # orphan I- — treat as new entity
            if cur: spans.append(cur)
            cur = [int(s), int(e), tag[2:]]
    if cur: spans.append(cur)
    return [(s, e, l, text[s:e]) for s, e, l in spans]

# --- Test on a handwritten dark-web-style snippet -------------------
test_text = ("New Fortinet SSL-VPN RCE — CVE-2024-21762 weaponized in the wild. "
             "Reports suggest APT28 and a Lockbit affiliate are chaining it with Cobalt Strike. "
             "Selling access to compromised hosts at mixer42xxx.onion, accepting BTC to "
             "bc1qxy2kgdygjrsqtzq2n0yrf2493p83kkfjhx0wlh. Email: broker@protonmail.com. "
             "Also have RDP dumps for Marriott and several Fortune 500 firms.")

print('Test extraction:\n')
for s, e, l, txt in extract_entities(test_text):
    print(f'  {l:<18s} {txt!r}')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to models/darkbert_entity_final
Test extraction:

  CRYPTO_WALLET      'bc1qxy2kgdygjrsqtzq2n0yrf2493p83kkfjhx0wlh'
  CREDENTIAL_TYPE    'RDP'
  CREDENTIAL_TYPE    'dumps'


## What you've done

- Built three labeling strategies (regex, GLiNER zero-shot, hybrid) and compared them *qualitatively* on CoDA — different methods cover different entity types, and the "best" one depends on what you're extracting.
- Produced **silver labels on the full English CoDA corpus** via the hybrid method — a reusable weak-supervision dataset.
- Fine-tuned **DarkBERT as a token classifier** on those silver labels, with overlapping chunks so long pages don't lose their tail entities.
- Ended up with a saved CTI entity extractor — ready to plug into the asset-matching pipeline in nb 06.

## Honest caveats

- **Silver, not gold.** The eval F1 measures recovery of the weak-supervision signal, not real-world extraction quality. A small hand-labeled test set would tell us the true ceiling — a good task for a follow-up.
- **GLiNER zero-shot is the bottleneck for fuzzy entities.** If it misses a threat actor, the silver labels miss it, and the fine-tuned model learns to miss it. Worth comparing against a proper CTI NER benchmark like CyNER or DNRTI.
- **Credential-type coverage is shallow.** We only match ~8 keywords. Extending this list (or training a GLiNER prompt for it) is the easiest quality win.

## What's next

- **Nb 06 — Asset matching.** Given a synthetic customer profile (domain, employee emails, brand terms), scan CoDA through this extractor and surface hits. That's the SOC-operational version of dark web monitoring.